# Application Feature Engineering

This notebook builds new applicant-level features from the cleaned application data. It creates affordability ratios, age and stability measures, external-score combinations, social-circle and enquiry counts, and a few document, contact, and region features.


## Import libraries


In [3]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 150)
print("Libraries imported successfully.")

Libraries imported successfully.


## Set project paths


In [5]:
current_folder = Path.cwd().resolve()
project_root = current_folder.parent if current_folder.name == "notebooks" else current_folder
input_path = project_root / "data" / "interim" / "application_clean.pkl"
training_ids_path = project_root / "data" / "modeling" / "splits" / "training_ids.csv"
test_ids_path = project_root / "data" / "modeling" / "splits" / "test_ids.csv"
output_path = project_root / "data" / "features" / "application_features.pkl"
audit_folder = project_root / "reports" / "audits"
output_path.parent.mkdir(parents=True, exist_ok=True)
audit_folder.mkdir(parents=True, exist_ok=True)
for required_path in [input_path, training_ids_path, test_ids_path]:
    assert required_path.exists(), f"Required file was not found: {required_path}"
print("Clean input:", input_path)
print("Feature output:", output_path)

Clean input: /Users/taranveersingh/A-MRP/data/interim/application_clean.pkl
Feature output: /Users/taranveersingh/A-MRP/data/features/application_features.pkl


## Load the cleaned application table


In [7]:
application_clean = pd.read_pickle(input_path)
training_ids = pd.read_csv(training_ids_path)["SK_ID_CURR"]
test_ids = pd.read_csv(test_ids_path)["SK_ID_CURR"]
training_id_set = set(training_ids)
test_id_set = set(test_ids)
application_features = application_clean.copy()
original_ids = application_clean["SK_ID_CURR"].copy()
original_target = application_clean["TARGET"].copy()
original_columns = set(application_clean.columns)
print("Input shape:", application_clean.shape)
print("Training-set rows:", application_clean["SK_ID_CURR"].isin(training_id_set).sum())
print("Test-set rows:", application_clean["SK_ID_CURR"].isin(test_id_set).sum())

Input shape: (307511, 76)
Training-set rows: 246008
Test-set rows: 61503


This is the same 307,511 applicants and training/test split used throughout the project.


## Define a safe ratio function


In [10]:
def safe_ratio(numerator, denominator):
    denominator = denominator.replace(0, np.nan)
    result = numerator / denominator
    return result.replace([np.inf, -np.inf], np.nan)

print("Safe ratio function created.")

Safe ratio function created.


## Create affordability and loan-structure features


In [12]:
application_features["APP_CREDIT_INCOME_RATIO"] = safe_ratio(application_features["AMT_CREDIT"], application_features["AMT_INCOME_TOTAL"])
application_features["APP_ANNUITY_INCOME_RATIO"] = safe_ratio(application_features["AMT_ANNUITY"], application_features["AMT_INCOME_TOTAL"])
application_features["APP_CREDIT_ANNUITY_RATIO"] = safe_ratio(application_features["AMT_CREDIT"], application_features["AMT_ANNUITY"])
application_features["APP_GOODS_CREDIT_RATIO"] = safe_ratio(application_features["AMT_GOODS_PRICE"], application_features["AMT_CREDIT"])
application_features["APP_CREDIT_GOODS_DIFFERENCE"] = application_features["AMT_CREDIT"] - application_features["AMT_GOODS_PRICE"]
application_features["APP_INCOME_PER_PERSON"] = safe_ratio(application_features["AMT_INCOME_TOTAL"], application_features["CNT_FAM_MEMBERS"])
application_features["APP_CHILDREN_FAMILY_RATIO"] = safe_ratio(application_features["CNT_CHILDREN"], application_features["CNT_FAM_MEMBERS"])
application_features["APP_HAS_CHILDREN"] = application_features["CNT_CHILDREN"].gt(0).astype("int8")
print("Affordability and loan features created: 8")

Affordability and loan features created: 8


These ratios describe how affordable the loan is relative to income, and how the loan amount compares with the price of the goods being financed.


## Create age and stability features


In [15]:
application_features["APP_AGE_YEARS"] = -application_features["DAYS_BIRTH"] / 365.25
application_features["APP_EMPLOYMENT_YEARS"] = -application_features["DAYS_EMPLOYED"] / 365.25
application_features["APP_REGISTRATION_YEARS"] = -application_features["DAYS_REGISTRATION"] / 365.25
application_features["APP_ID_PUBLISH_YEARS"] = -application_features["DAYS_ID_PUBLISH"] / 365.25
application_features["APP_PHONE_CHANGE_YEARS"] = -application_features["DAYS_LAST_PHONE_CHANGE"] / 365.25
application_features["APP_EMPLOYED_AGE_RATIO"] = safe_ratio(application_features["DAYS_EMPLOYED"], application_features["DAYS_BIRTH"])
application_features["APP_REGISTRATION_AGE_RATIO"] = safe_ratio(application_features["DAYS_REGISTRATION"], application_features["DAYS_BIRTH"])
application_features["APP_ID_AGE_RATIO"] = safe_ratio(application_features["DAYS_ID_PUBLISH"], application_features["DAYS_BIRTH"])
print("Age and stability features created: 8")

Age and stability features created: 8


Age, employment, and a few other date columns are converted from days to years to make them easier to interpret, and a few ratios compare these against age.


## Create external-score agreement features


In [18]:
application_features["APP_EXT_2_3_MEAN"] = application_features[["EXT_SOURCE_2", "EXT_SOURCE_3"]].mean(axis=1)
application_features["APP_EXT_2_3_MIN"] = application_features[["EXT_SOURCE_2", "EXT_SOURCE_3"]].min(axis=1)
application_features["APP_EXT_2_3_MAX"] = application_features[["EXT_SOURCE_2", "EXT_SOURCE_3"]].max(axis=1)
application_features["APP_EXT_2_3_GAP"] = (application_features["EXT_SOURCE_2"] - application_features["EXT_SOURCE_3"]).abs()
application_features["APP_EXT_2_3_PRODUCT"] = application_features["EXT_SOURCE_2"] * application_features["EXT_SOURCE_3"]
application_features["APP_EXT_MEAN_SQUARED"] = application_features["EXT_SOURCE_MEAN"] ** 2
print("External-score features created: 6")

External-score features created: 6


These combine EXT_SOURCE_2 and EXT_SOURCE_3 in different ways (mean, min, max, gap, product), since these two are the more complete external scores. EXT_SOURCE_MEAN squared is also added as a simple non-linear version of the existing mean feature.


## Create social-circle and enquiry features


In [21]:
application_features["APP_SOCIAL_OBS_TOTAL"] = application_features[["OBS_30_CNT_SOCIAL_CIRCLE", "OBS_60_CNT_SOCIAL_CIRCLE"]].sum(axis=1, min_count=1)
application_features["APP_SOCIAL_DEF_TOTAL"] = application_features[["DEF_30_CNT_SOCIAL_CIRCLE", "DEF_60_CNT_SOCIAL_CIRCLE"]].sum(axis=1, min_count=1)

# A zero observed circle has zero observed defaults. Genuine source missingness stays missing.
social_ratio_denominator = application_features["APP_SOCIAL_OBS_TOTAL"].replace(0, 1)
application_features["APP_SOCIAL_DEFAULT_RATIO"] = safe_ratio(
    application_features["APP_SOCIAL_DEF_TOTAL"], social_ratio_denominator
)

enquiry_columns = [c for c in application_features.columns if c.startswith("AMT_REQ_CREDIT_BUREAU_")]
application_features["APP_CREDIT_ENQUIRY_TOTAL"] = application_features[enquiry_columns].sum(axis=1, min_count=1)
recent_enquiry_columns = [c for c in ["AMT_REQ_CREDIT_BUREAU_HOUR", "AMT_REQ_CREDIT_BUREAU_DAY", "AMT_REQ_CREDIT_BUREAU_WEEK", "AMT_REQ_CREDIT_BUREAU_MON", "AMT_REQ_CREDIT_BUREAU_QRT"] if c in application_features.columns]
application_features["APP_RECENT_ENQUIRY_TOTAL"] = application_features[recent_enquiry_columns].sum(axis=1, min_count=1)
application_features["APP_HAS_RECENT_ENQUIRY"] = application_features["APP_RECENT_ENQUIRY_TOTAL"].gt(0).astype("int8")
print("Social and enquiry features created: 6")

Social and enquiry features created: 6


The social-circle ratio treats a zero observed circle as a zero default rate rather than missing, since there is nothing to observe there. Genuine missing values from the original data stay missing.


## Create document, contact, region and address features


In [24]:
document_columns = [c for c in application_features.columns if c.startswith("FLAG_DOCUMENT_")]
contact_columns = [c for c in ["FLAG_MOBIL", "FLAG_EMP_PHONE", "FLAG_WORK_PHONE", "FLAG_CONT_MOBILE", "FLAG_PHONE", "FLAG_EMAIL"] if c in application_features.columns]
address_columns = [c for c in application_features.columns if c.startswith("REG_") and "NOT_" in c]
region_rating_columns = [c for c in ["REGION_RATING_CLIENT", "REGION_RATING_CLIENT_W_CITY"] if c in application_features.columns]

application_features["APP_DOCUMENT_COUNT"] = application_features[document_columns].sum(axis=1)
application_features["APP_CONTACT_COUNT"] = application_features[contact_columns].sum(axis=1)
application_features["APP_ADDRESS_MISMATCH_COUNT"] = application_features[address_columns].sum(axis=1)
application_features["APP_REGION_RATING_MEAN"] = application_features[region_rating_columns].mean(axis=1)
application_features["APP_WEEKEND_APPLICATION"] = application_features["WEEKDAY_APPR_PROCESS_START"].isin(["SATURDAY", "SUNDAY"]).astype("int8")
application_features["APP_BUSINESS_HOURS_APPLICATION"] = application_features["HOUR_APPR_PROCESS_START"].between(9, 17).astype("int8")
print("Document, contact, region and timing features created: 6")

Document, contact, region and timing features created: 6


These count things like how many optional documents were submitted and how many contact methods are on file, and flag whether the application was made on a weekend or outside business hours.


## Audit new-feature missingness and training-only target association


In [27]:
new_features = [c for c in application_features.columns if c not in original_columns]
training_data = application_features.loc[application_features["SK_ID_CURR"].isin(training_id_set)]
association_rows = []
for feature in new_features:
    series = training_data[feature]
    correlation = series.corr(training_data["TARGET"]) if series.nunique(dropna=True) > 1 else np.nan
    association_rows.append({
        "feature": feature,
        "missing_count": int(series.isna().sum()),
        "missing_rate": series.isna().mean(),
        "unique_non_missing": int(series.nunique(dropna=True)),
        "pearson_target_correlation": correlation,
        "absolute_correlation": abs(correlation) if pd.notna(correlation) else np.nan,
        "selection_decision": "Keep until global feature-selection comparison",
    })
feature_audit = pd.DataFrame(association_rows).sort_values("absolute_correlation", ascending=False).reset_index(drop=True)
print("New features created:", len(new_features))
feature_audit.round(5)

New features created: 34


,feature,missing_count,missing_rate,unique_non_missing,pearson_target_correlation,absolute_correlation,selection_decision
0,APP_EXT_2_3_MEAN,192,0.00078,234594,-0.21089,0.21089,Keep until global feature-selection comparison
1,APP_EXT_MEAN_SQUARED,144,0.00059,241053,-0.20402,0.20402,Keep until global feature-selection comparison
2,APP_EXT_2_3_PRODUCT,49237,0.20014,195594,-0.19968,0.19968,Keep until global feature-selection comparison
3,APP_EXT_2_3_MAX,192,0.00078,77505,-0.19074,0.19074,Keep until global feature-selection comparison
4,APP_EXT_2_3_MIN,192,0.00078,87804,-0.18552,0.18552,Keep until global feature-selection comparison
5,APP_AGE_YEARS,0,0.00000,17411,-0.07927,0.07927,Keep until global feature-selection comparison
6,APP_EMPLOYMENT_YEARS,44283,0.18001,12059,-0.07587,0.07587,Keep until global feature-selection comparison
7,APP_EMPLOYED_AGE_RATIO,44283,0.18001,199947,-0.06893,0.06893,Keep until global feature-selection comparison
8,APP_GOODS_CREDIT_RATIO,220,0.00089,2852,-0.06497,0.06497,Keep until global feature-selection comparison
9,APP_REGION_RATING_MEAN,0,0.00000,5,0.05978,0.05978,Keep until global feature-selection comparison


This checks the missing rate and correlation with default for every new feature, but only using the training applicants, so the final test set is not touched at this stage.


## Validate engineered data


In [30]:
numeric_columns = application_features.select_dtypes(include="number").columns
infinite_count = sum(int(np.isinf(application_features[c].dropna()).sum()) for c in numeric_columns)
social_ratio_missing_rate = training_data["APP_SOCIAL_DEFAULT_RATIO"].isna().mean()
validation_checks = pd.DataFrame([
    {"check": "Row count unchanged", "passed": len(application_features) == len(application_clean)},
    {"check": "Applicant IDs unchanged", "passed": application_features["SK_ID_CURR"].equals(original_ids)},
    {"check": "Applicant IDs unique", "passed": application_features["SK_ID_CURR"].is_unique},
    {"check": "TARGET unchanged", "passed": application_features["TARGET"].equals(original_target)},
    {"check": "Training and test IDs separate", "passed": len(training_id_set.intersection(test_id_set)) == 0},
    {"check": "New feature names unique", "passed": len(new_features) == len(set(new_features))},
    {"check": "No infinite numerical values", "passed": infinite_count == 0},
    {"check": "Social-default ratio below 50 percent missing", "passed": social_ratio_missing_rate < 0.50},
    {"check": "All new features included in audit", "passed": set(new_features) == set(feature_audit["feature"])},
    {"check": "Final test excluded from associations", "passed": not training_data["SK_ID_CURR"].isin(test_id_set).any()},
])
assert validation_checks["passed"].all(), "At least one application feature-engineering check failed."
validation_checks

,check,passed
0,Row count unchanged,True
1,Applicant IDs unchanged,True
2,Applicant IDs unique,True
3,TARGET unchanged,True
4,Training and test IDs separate,True
5,New feature names unique,True
6,No infinite numerical values,True
7,Social-default ratio below 50 percent missing,True
8,All new features included in audit,True
9,Final test excluded from associations,True


All checks passed.


## Save features and audit reports


In [33]:
application_features.to_pickle(output_path)
feature_audit.to_csv(audit_folder / "application_engineered_feature_audit.csv", index=False)
validation_checks.to_csv(audit_folder / "application_feature_engineering_validation.csv", index=False)
print("Application feature table saved:", output_path)
print("Output rows:", len(application_features))
print("Output columns:", application_features.shape[1])
print("New features created:", len(new_features))
print("Remaining numerical missing values:", int(application_features.select_dtypes(include="number").isna().sum().sum()))

Application feature table saved: /Users/taranveersingh/A-MRP/data/features/application_features.pkl
Output rows: 307511
Output columns: 110
New features created: 34
Remaining numerical missing values: 1749260


## Main feature engineering results

This notebook added 34 new applicant-level features to the cleaned application data, covering affordability, age and stability, external-score combinations, social-circle behaviour, and a few document and contact counts.

All checks passed: the row count, applicant IDs, and target are unchanged, and no infinite values were introduced. The feature table now has 110 columns for all 307,511 applicants. The next step is to build features from the bureau data.
